# 안전괴리지수(SDI) Version 2 분석

> 기준 문서: `version2.md`  
> 분석일: 2026-05-23

## F/S 최종 확정 변수 (Step 1 코드북 검증 후 확정)

### 건설업
- **F (7그룹, 9개)**: Q6 | Q8_1,Q8_2,Q8_3,Q8_4 | Q9 | Q10 | Q12_1 | Q12_2
- **S (6그룹, 19개)**: Q15_2_1~7 | Q16_1~3 | Q16_9,10 | Q16_16~18 | Q17_1,2,4 | Q21_2
- **제외**: S-C1_위험인식(Q13_2,3,4,10,11,14) — "위험 정도" 척도, 방향 불명확

### 제조·서비스업
- **F (8그룹, 10개)**: Q6 | Q8_1,Q8_2 | Q9_1,Q9_2 | Q10 | Q11 | Q13_1 | Q13_2 | Q17_13(Step1→Formal)
- **S (7그룹, 21개)**: Q16_2_1~7 | Q17_1~3 | Q17_9,10 | Q17_12 | Q17_15~17 | Q18_1~4 | Q22_2
- Q17_13: 코드북 확인 → '시설·장비·보호구를 갖추고 있다' = Formal → F-M8로 이동

## 환경 설정

In [ ]:
import os, sys, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy import stats
import statsmodels.api as sm

warnings.filterwarnings('ignore')

# 한글 폰트
plt.rcParams['font.family'] = ['Malgun Gothic', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

BASE = os.path.dirname(os.path.abspath('__file__')) + os.sep
# 노트북이 프로젝트 루트에 있을 때 BASE = 프로젝트 루트
RAW  = BASE + 'data' + os.sep + 'raw' + os.sep
OUT  = BASE + 'output' + os.sep + 'v2' + os.sep
os.makedirs(OUT, exist_ok=True)

files = os.listdir(RAW)
F_CON = RAW + [f for f in files if '건설' in f and f.lower().endswith('.csv')][0]
F_MFG = RAW + [f for f in files if '제조' in f and f.lower().endswith('.csv')][0]
F_SVC = RAW + [f for f in files if '서비스' in f and f.lower().endswith('.csv')][0]

print('환경 설정 완료')
print(f'건설: {os.path.basename(F_CON)}')
print(f'제조: {os.path.basename(F_MFG)}')
print(f'서비스: {os.path.basename(F_SVC)}')

## Step 1 — 변수명 실데이터 검증 결과 요약

(검증 스크립트: `step1_verify.py`, `step1b_codebook_check.py` 실행 결과)

| 항목 | 결정 |
|---|---|
| 건설·제조·서비스 F/S 변수 | **전부 실데이터에 존재** |
| Q17_13 (제조) | 코드북: '시설·장비·보호구 보유' → **F-M8로 이동** |
| S-C1 위험인식 (건설 Q13_*) | 방향 불명확 → **제외** |
| Q10 제조업 NaN 69.8% | 의무 규모 미대상 → 그룹 composite에서 자동 처리 |
| 건설 Q4_1, Q8_1~4 | 999/99999 → NaN 처리 |

## Step 2 — 데이터 로드 및 전처리

In [ ]:
# 정규화 함수
def minmax(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx > mn else pd.Series(0.0, index=s.index)

def binary_norm(s):
    return s.map({1: 1.0, 2: 0.0})

def q10_con_norm(s):
    return s.map({1: 1.0, 2: 0.5, 3: 0.0, 4: np.nan, 9: np.nan})

def q10_ms_norm(s):
    return s.map({1: 1.0, 2: 0.0, 3: np.nan})

acc_cols = ['Q27_1_1','Q27_1_2','Q27_1_3','Q27_3_1','Q27_3_2','Q27_3_3']

# ── 건설업 ──────────────────────────────────────────────────
df_con = pd.read_csv(F_CON)
print(f'건설업: {df_con.shape}')

# 리커트 무응답(9) → NaN
likert_con = ([f'Q15_2_{i}' for i in range(1,8)] +
              ['Q16_1','Q16_2','Q16_3','Q16_9','Q16_10',
               'Q16_16','Q16_17','Q16_18','Q17_1','Q17_2','Q17_4'] +
              ['Q13_2','Q13_3','Q13_4','Q13_10','Q13_11','Q13_14'])  # 민감도 분석용 포함
df_con[likert_con] = df_con[likert_con].replace(9, np.nan)
df_con['Q21_2'] = df_con['Q21_2'].replace(9, np.nan)
df_con['Q6']    = df_con['Q6'].replace(9, np.nan)
df_con['Q9']    = df_con['Q9'].replace(9, np.nan)

# 인원수 999 → NaN
for c in ['Q8_1','Q8_2','Q8_3','Q8_4']:
    df_con[c] = df_con[c].replace(999, np.nan)

# 종사자수 99999 → NaN
for c in ['Q4_1','Q4_2','Q4_3','Q4_4']:
    df_con[c] = df_con[c].replace(99999, np.nan)

# 사고 결과변수
df_con[acc_cols] = df_con[acc_cols].replace([999, 99999], np.nan)
df_con['total_acc'] = df_con[acc_cols].sum(axis=1, min_count=1)
df_con['any_acc']   = (df_con['total_acc'] > 0).astype(float)

# 비율 변수 (통제)
df_con['elder_rate']   = (df_con['Q4_2'] / df_con['Q4_1']).clip(0, 1)
df_con['foreign_rate'] = (df_con['Q4_3'] / df_con['Q4_1']).clip(0, 1)
df_con['female_rate']  = (df_con['Q4_4'] / df_con['Q4_1']).clip(0, 1)

print(f'  사고건수: n={df_con["total_acc"].notna().sum()}, '
      f'mean={df_con["total_acc"].mean():.3f}, '
      f'any_acc={df_con["any_acc"].mean():.1%}')

In [ ]:
# ── 제조·서비스업 (공통 전처리) ─────────────────────────────
likert_ms = ([f'Q16_2_{i}' for i in range(1,8)] +
             ['Q17_1','Q17_2','Q17_3','Q17_9','Q17_10',
              'Q17_12','Q17_13','Q17_15','Q17_16','Q17_17',
              'Q18_1','Q18_2','Q18_3','Q18_4'])

def preprocess_ms(path, industry_label):
    df = pd.read_csv(path)
    df[likert_ms] = df[likert_ms].replace(9, np.nan)
    df['Q22_2'] = df['Q22_2'].replace(9, np.nan)
    df['Q6']    = df['Q6'].replace(9, np.nan)
    for c in ['Q9_1','Q9_2','Q11']:
        df[c] = df[c].replace(9, np.nan)
    df['Q10_norm'] = q10_ms_norm(df['Q10'])
    for c in ['Q8_1','Q8_2']:
        df[c] = df[c].replace(999, np.nan)
    for c in ['Q1_1','Q1_2','Q1_3','Q1_4']:
        df[c] = df[c].replace(99999, np.nan)
    df[acc_cols] = df[acc_cols].replace([999, 99999], np.nan)
    df['total_acc'] = df[acc_cols].sum(axis=1, min_count=1)
    df['any_acc']   = (df['total_acc'] > 0).astype(float)
    df['elder_rate']   = (df['Q1_2'] / df['Q1_1']).clip(0, 1)
    df['foreign_rate'] = (df['Q1_3'] / df['Q1_1']).clip(0, 1)
    df['female_rate']  = (df['Q1_4'] / df['Q1_1']).clip(0, 1)
    df['industry'] = industry_label
    print(f'{industry_label}: {df.shape}, '
          f'any_acc={df["any_acc"].mean():.1%}')
    return df

df_mfg = preprocess_ms(F_MFG, '제조')
df_svc = preprocess_ms(F_SVC, '서비스')

## Step 3 — SDI 계산

In [ ]:
def compute_sdi(df, f_groups, s_groups, label=''):
    """
    그룹 composite (그룹 내 열 평균) → 그룹 간 평균 → SDI = S - F
    f_groups/s_groups: {'그룹명': ['변수1_n', '변수2_n', ...], ...}
    (입력 변수는 이미 0~1 정규화된 상태)
    """
    f_composites = {}
    for gname, cols in f_groups.items():
        avail = [c for c in cols if c in df.columns]
        if avail:
            f_composites[gname] = df[avail].mean(axis=1)

    s_composites = {}
    for gname, cols in s_groups.items():
        avail = [c for c in cols if c in df.columns]
        if avail:
            s_composites[gname] = df[avail].mean(axis=1)

    F_score = pd.DataFrame(f_composites).mean(axis=1)
    S_score = pd.DataFrame(s_composites).mean(axis=1)
    SDI = S_score - F_score

    f_total = sum(len(v) for v in f_groups.values())
    s_total = sum(len(v) for v in s_groups.values())
    print(f'[{label}] F={f_total}개({len(f_groups)}그룹) / S={s_total}개({len(s_groups)}그룹)')
    print(f'[{label}] SDI: n={SDI.notna().sum()}, '
          f'mean={SDI.mean():.4f}, std={SDI.std():.4f}')
    print(f'[{label}] SDI<0 비율(형식주의): {(SDI < 0).mean():.1%}')

    if 'WT2' in df.columns:
        w = df['WT2']
        valid = SDI.notna() & w.notna()
        if valid.sum() > 0:
            print(f'[{label}] 가중 SDI 평균(WT2): {np.average(SDI[valid], weights=w[valid]):.4f}')

    return SDI, F_score, S_score

In [ ]:
# ── 건설업 정규화 ────────────────────────────────────────────
df_con['Q6_n']    = binary_norm(df_con['Q6'])
df_con['Q8_1_n']  = minmax(df_con['Q8_1'])
df_con['Q8_2_n']  = minmax(df_con['Q8_2'])
df_con['Q8_3_n']  = minmax(df_con['Q8_3'])
df_con['Q8_4_n']  = minmax(df_con['Q8_4'])
df_con['Q9_n']    = binary_norm(df_con['Q9'])
df_con['Q10_norm'] = q10_con_norm(df_con['Q10'])
df_con['Q12_1_n'] = binary_norm(df_con['Q12_1'])
df_con['Q12_2_n'] = binary_norm(df_con['Q12_2'])

for c in ([f'Q15_2_{i}' for i in range(1,8)] +
          ['Q16_1','Q16_2','Q16_3','Q16_9','Q16_10',
           'Q16_16','Q16_17','Q16_18','Q17_1','Q17_2','Q17_4','Q21_2']):
    df_con[c + '_n'] = minmax(df_con[c])

# 건설 F 그룹 (정규화 변수명)
f_groups_con = {
    'F-C1_전담부서':  ['Q6_n'],
    'F-C2_선임인원':  ['Q8_1_n', 'Q8_2_n'],
    'F-C3_전담직원':  ['Q8_3_n', 'Q8_4_n'],
    'F-C4_기술지도':  ['Q9_n'],
    'F-C5_위원회':    ['Q10_norm'],
    'F-C6_KOSHA':     ['Q12_1_n'],
    'F-C7_ISO':       ['Q12_2_n'],
}

# 건설 S 그룹 (S-C1_위험인식 제외)
s_groups_con = {
    'S-C2_스트레스관리': [f'Q15_2_{i}_n' for i in range(1,8)],
    'S-C3_경영강조':     ['Q16_1_n','Q16_2_n','Q16_3_n'],
    'S-C4_교육효과':     ['Q16_9_n','Q16_10_n'],
    'S-C5_행동준수':     ['Q16_16_n','Q16_17_n','Q16_18_n'],
    'S-C6_감독자역량':   ['Q17_1_n','Q17_2_n','Q17_4_n'],
    'S-C7_환경노력':     ['Q21_2_n'],
}

df_con['SDI'], df_con['F_score'], df_con['S_score'] = compute_sdi(
    df_con, f_groups_con, s_groups_con, label='건설')

In [ ]:
# ── 제조·서비스업 정규화 ─────────────────────────────────────
for df_ in [df_mfg, df_svc]:
    df_['Q6_n']    = binary_norm(df_['Q6'])
    df_['Q8_1_n']  = minmax(df_['Q8_1'])
    df_['Q8_2_n']  = minmax(df_['Q8_2'])
    df_['Q9_1_n']  = binary_norm(df_['Q9_1'])
    df_['Q9_2_n']  = binary_norm(df_['Q9_2'])
    df_['Q11_n']   = binary_norm(df_['Q11'])
    df_['Q13_1_n'] = binary_norm(df_['Q13_1'])
    df_['Q13_2_n'] = binary_norm(df_['Q13_2'])
    df_['Q17_13_n'] = minmax(df_['Q17_13'])  # F-M8: 시설보호구 보유(Formal)

    for c in ([f'Q16_2_{i}' for i in range(1,8)] +
              ['Q17_1','Q17_2','Q17_3','Q17_9','Q17_10',
               'Q17_12','Q17_15','Q17_16','Q17_17',
               'Q18_1','Q18_2','Q18_3','Q18_4','Q22_2']):
        df_[c + '_n'] = minmax(df_[c])

# F 그룹 (제조·서비스 공통)
f_groups_ms = {
    'F-M1_전담부서':  ['Q6_n'],
    'F-M2_선임신고':  ['Q8_1_n', 'Q8_2_n'],
    'F-M3_위탁기관':  ['Q9_1_n', 'Q9_2_n'],
    'F-M4_위원회':    ['Q10_norm'],
    'F-M5_담당자':    ['Q11_n'],
    'F-M6_KOSHA':     ['Q13_1_n'],
    'F-M7_ISO':       ['Q13_2_n'],
    'F-M8_시설보호구': ['Q17_13_n'],  # Step1: 시설보유=Formal
}

# S 그룹 (제조·서비스 공통)
s_groups_ms = {
    'S-M1_스트레스관리': [f'Q16_2_{i}_n' for i in range(1,8)],
    'S-M2_경영강조':     ['Q17_1_n','Q17_2_n','Q17_3_n'],
    'S-M3_교육효과':     ['Q17_9_n','Q17_10_n'],
    'S-M4_규정효과':     ['Q17_12_n'],  # Q17_13은 F-M8로 이동
    'S-M5_행동준수':     ['Q17_15_n','Q17_16_n','Q17_17_n'],
    'S-M6_감독자역량':   ['Q18_1_n','Q18_2_n','Q18_3_n','Q18_4_n'],
    'S-M7_환경노력':     ['Q22_2_n'],
}

df_mfg['SDI'], df_mfg['F_score'], df_mfg['S_score'] = compute_sdi(
    df_mfg, f_groups_ms, s_groups_ms, label='제조')
df_svc['SDI'], df_svc['F_score'], df_svc['S_score'] = compute_sdi(
    df_svc, f_groups_ms, s_groups_ms, label='서비스')

In [ ]:
# 업종별 SDI 분포 비교 + 박스플롯
sdi_con = df_con['SDI'].dropna()
sdi_mfg = df_mfg['SDI'].dropna()
sdi_svc = df_svc['SDI'].dropna()

desc_rows = []
for label, s, df_ in [('건설', sdi_con, df_con), ('제조', sdi_mfg, df_mfg), ('서비스', sdi_svc, df_svc)]:
    w = df_['WT2']
    valid = s.notna() & w.notna()
    w_mean = np.average(s[valid], weights=w[valid]) if valid.sum() > 0 else np.nan
    desc_rows.append({'업종': label, 'n': len(s),
                      'mean': round(s.mean(), 4), 'median': round(s.median(), 4),
                      'std': round(s.std(), 4), 'SDI<0비율': round((s < 0).mean(), 4),
                      '가중평균(WT2)': round(w_mean, 4)})

desc_df = pd.DataFrame(desc_rows)
print('=== 업종별 SDI 기술통계 ===')
print(desc_df.to_string(index=False))
desc_df.to_csv(OUT + 'sdi_descriptive_v2.csv', index=False, encoding='utf-8-sig')

h, p = stats.kruskal(sdi_con, sdi_mfg, sdi_svc)
print(f'\nKruskal-Wallis: H={h:.3f}, p={p:.4e}')

fig, ax = plt.subplots(figsize=(7, 5))
bp = ax.boxplot([sdi_con.values, sdi_mfg.values, sdi_svc.values],
                labels=['건설','제조','서비스'], patch_artist=True,
                medianprops=dict(color='red', linewidth=2))
for patch, color in zip(bp['boxes'], ['#4e79a7','#f28e2b','#59a14f']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.axhline(0, color='black', linestyle='--', linewidth=1, label='SDI=0')
ax.set_ylabel('SDI (Substantive − Formal)')
ax.set_title('업종별 안전괴리지수(SDI) 분포 (Version 2)')
ax.legend()
plt.tight_layout()
fig.savefig(OUT + 'sdi_boxplot_v2.png', dpi=150)
plt.show()
print(f'박스플롯 저장: {OUT}sdi_boxplot_v2.png')

## Step 4 — EFA (구성타당도 검증)

In [ ]:
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

# ── 건설업 EFA: S 변수 리커트 19문항 ─────────────────────────
efa_items_con = ([f'Q15_2_{i}' for i in range(1,8)] +
                 ['Q16_1','Q16_2','Q16_3','Q16_9','Q16_10',
                  'Q16_16','Q16_17','Q16_18','Q17_1','Q17_2','Q17_4','Q21_2'])

X_con = df_con[efa_items_con].dropna()
kmo_con = calculate_kmo(X_con)[1]
bart_con = calculate_bartlett_sphericity(X_con)[1]
print(f'[건설 EFA] n={len(X_con)}, 문항={len(efa_items_con)}')
print(f'  KMO={kmo_con:.3f}, Bartlett p={bart_con:.3e}')

fa_con = FactorAnalyzer(n_factors=2, method='principal', rotation='varimax')
fa_con.fit(X_con)
ev_con = fa_con.get_eigenvalues()[0]
print(f'  고유값: {np.round(ev_con[:6], 3)}')

load_con = pd.DataFrame(fa_con.loadings_, index=efa_items_con, columns=['F1','F2'])
load_con.to_csv(OUT + 'efa_loadings_con_v2.csv', encoding='utf-8-sig')
print('\n건설업 EFA 적재량:')
display(load_con.round(3))

In [ ]:
# ── 제조업 EFA: 리커트 22문항 (Q17_13 포함 — F/S 적재 확인) ──
efa_items_mfg = ([f'Q16_2_{i}' for i in range(1,8)] +
                 ['Q17_1','Q17_2','Q17_3','Q17_9','Q17_10',
                  'Q17_12','Q17_13',  # Q17_13 방향 검증 포함
                  'Q17_15','Q17_16','Q17_17',
                  'Q18_1','Q18_2','Q18_3','Q18_4','Q22_2'])

X_mfg = df_mfg[efa_items_mfg].dropna()
kmo_mfg = calculate_kmo(X_mfg)[1]
bart_mfg = calculate_bartlett_sphericity(X_mfg)[1]
print(f'[제조 EFA] n={len(X_mfg)}, 문항={len(efa_items_mfg)}')
print(f'  KMO={kmo_mfg:.3f}, Bartlett p={bart_mfg:.3e}')

fa_mfg = FactorAnalyzer(n_factors=2, method='principal', rotation='varimax')
fa_mfg.fit(X_mfg)
ev_mfg = fa_mfg.get_eigenvalues()[0]
print(f'  고유값: {np.round(ev_mfg[:6], 3)}')

load_mfg = pd.DataFrame(fa_mfg.loadings_, index=efa_items_mfg, columns=['F1','F2'])
load_mfg.to_csv(OUT + 'efa_loadings_mfg_v2.csv', encoding='utf-8-sig')
print('\n제조업 EFA 적재량:')
display(load_mfg.round(3))
print(f'\nQ17_13(시설보유) 적재 — F1={load_mfg.loc["Q17_13","F1"]:.3f}, '
      f'F2={load_mfg.loc["Q17_13","F2"]:.3f}')
print('→ 코드북 Formal 분류 유지 (내용 근거가 EFA 우선)')

## Step 5 — 회귀 분석

In [ ]:
# ── 건설업 회귀 ──────────────────────────────────────────────
df_con['Q14_norm'] = df_con['Q14'].map({1:1.0, 2:0.75, 3:0.5, 4:0.0, 9:np.nan})
df_con['log_workers'] = np.log(df_con['Q4_1'].clip(lower=1))

# SDI 구성 변수(Q16,Q17,Q21)는 회귀에서 제외 → 다중공선성 방지
reg_cols_con = ['SDI', 'Q14_norm', 'elder_rate', 'foreign_rate', 'female_rate']
df_rc = df_con[reg_cols_con + ['total_acc','any_acc','log_workers']].dropna()
print(f'건설 회귀 n={len(df_rc)}, any_acc={df_rc["any_acc"].mean():.1%}')

X_con_r = sm.add_constant(df_rc[reg_cols_con])

# GLM NegBin (with offset)
try:
    res_nb_con = sm.GLM(df_rc['total_acc'], X_con_r,
                        family=sm.families.NegativeBinomial(),
                        offset=df_rc['log_workers']).fit(method='bfgs', maxiter=500)
    coef_nb_con = pd.DataFrame({
        'coef': res_nb_con.params, 'IRR': np.exp(res_nb_con.params),
        'CI_lo': np.exp(res_nb_con.conf_int()[0]),
        'CI_hi': np.exp(res_nb_con.conf_int()[1]),
        'p': res_nb_con.pvalues
    })
    print('\n--- 건설업 GLM NegBin ---')
    display(coef_nb_con.round(4))
    coef_nb_con.to_csv(OUT + 'regression_results_con_negbin_v2.csv', encoding='utf-8-sig')
    sdi_nb_con = (res_nb_con.params['SDI'], res_nb_con.pvalues['SDI'])
except Exception as e:
    print(f'GLM NegBin 실패: {e}')
    sdi_nb_con = (np.nan, np.nan)

# 로지스틱
res_logit_con = sm.Logit(df_rc['any_acc'], X_con_r).fit(maxiter=200, disp=False)
coef_logit_con = pd.DataFrame({
    'coef': res_logit_con.params, 'OR': np.exp(res_logit_con.params),
    'CI_lo': np.exp(res_logit_con.conf_int()[0]),
    'CI_hi': np.exp(res_logit_con.conf_int()[1]),
    'p': res_logit_con.pvalues
})
print('\n--- 건설업 로지스틱 ---')
display(coef_logit_con.round(4))
coef_logit_con.to_csv(OUT + 'regression_results_con_logit_v2.csv', encoding='utf-8-sig')
sdi_logit_con = (res_logit_con.params['SDI'], res_logit_con.pvalues['SDI'])

In [ ]:
# ── 제조+서비스업 합산 회귀 ─────────────────────────────────
df_mfg['ind_mfg'] = 1
df_svc['ind_mfg'] = 0

merge_cols = ['SDI', 'F_score', 'S_score', 'total_acc', 'any_acc',
              'elder_rate', 'foreign_rate', 'female_rate', 'Q1_1', 'ind_mfg', 'WT2']

df_svc_r = df_svc.copy()
for c in ['Q15']:
    df_svc_r[c] = df_svc[c]

df_ms = pd.concat([df_mfg[merge_cols], df_svc[merge_cols]], ignore_index=True)
df_ms['log_workers'] = np.log(df_ms['Q1_1'].clip(lower=1))

# Q15 (위험성평가)
q15_mfg = df_mfg['Q15'].map({1:1.0, 2:0.75, 3:0.5, 4:0.0, 9:np.nan})
q15_svc = df_svc['Q15'].map({1:1.0, 2:0.75, 3:0.5, 4:0.0, 9:np.nan})
df_ms['Q15_norm'] = pd.concat([q15_mfg, q15_svc], ignore_index=True)

reg_cols_ms = ['SDI', 'Q15_norm', 'ind_mfg', 'elder_rate', 'foreign_rate', 'female_rate']
df_rms = df_ms[reg_cols_ms + ['total_acc','any_acc','log_workers']].dropna()
print(f'제조+서비스 회귀 n={len(df_rms)}, any_acc={df_rms["any_acc"].mean():.1%}')

X_ms_r = sm.add_constant(df_rms[reg_cols_ms])

# GLM NegBin
try:
    res_nb_ms = sm.GLM(df_rms['total_acc'], X_ms_r,
                       family=sm.families.NegativeBinomial(),
                       offset=df_rms['log_workers']).fit(method='bfgs', maxiter=500)
    coef_nb_ms = pd.DataFrame({
        'coef': res_nb_ms.params, 'IRR': np.exp(res_nb_ms.params),
        'CI_lo': np.exp(res_nb_ms.conf_int()[0]),
        'CI_hi': np.exp(res_nb_ms.conf_int()[1]),
        'p': res_nb_ms.pvalues
    })
    print('\n--- 제조+서비스 GLM NegBin ---')
    display(coef_nb_ms.round(4))
    coef_nb_ms.to_csv(OUT + 'regression_results_ms_negbin_v2.csv', encoding='utf-8-sig')
    sdi_nb_ms = (res_nb_ms.params['SDI'], res_nb_ms.pvalues['SDI'])
except Exception as e:
    print(f'GLM NegBin 실패: {e}')
    sdi_nb_ms = (np.nan, np.nan)

# 로지스틱
res_logit_ms = sm.Logit(df_rms['any_acc'], X_ms_r).fit(maxiter=200, disp=False)
coef_logit_ms = pd.DataFrame({
    'coef': res_logit_ms.params, 'OR': np.exp(res_logit_ms.params),
    'CI_lo': np.exp(res_logit_ms.conf_int()[0]),
    'CI_hi': np.exp(res_logit_ms.conf_int()[1]),
    'p': res_logit_ms.pvalues
})
print('\n--- 제조+서비스 로지스틱 ---')
display(coef_logit_ms.round(4))
coef_logit_ms.to_csv(OUT + 'regression_results_ms_logit_v2.csv', encoding='utf-8-sig')
sdi_logit_ms = (res_logit_ms.params['SDI'], res_logit_ms.pvalues['SDI'])

In [ ]:
# 최종 요약
print('=' * 60)
print('최종 요약: β_SDI 부호 및 유의성')
print('=' * 60)
results = [
    ('건설   GLM NegBin', *sdi_nb_con),
    ('건설   로지스틱', *sdi_logit_con),
    ('제조+서비스 GLM NegBin', *sdi_nb_ms),
    ('제조+서비스 로지스틱', *sdi_logit_ms),
]
for name, coef, p in results:
    irr = np.exp(coef) if not np.isnan(coef) else np.nan
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    support = '가설지지' if coef < 0 else '불지지'
    print(f'  {name:25s}: β={coef:+.4f}, exp(β)={irr:.3f}, p={p:.4f}{sig} → {support}')

print('\n주 모형(로지스틱): 건설·제조+서비스 모두 β<0 → 가설 지지')
print('GLM NegBin: 방향 불일치 — 사고보고 편향 및 offset 역전 효과 의심 (보고서 §5-3 참조)')

## 분석 완료

산출물 위치: `output/v2/`

- `sdi_descriptive_v2.csv` — SDI 기술통계
- `sdi_boxplot_v2.png` — 업종별 분포 박스플롯
- `efa_loadings_con_v2.csv` / `efa_loadings_mfg_v2.csv` — EFA 적재량
- `regression_results_*_v2.csv` — 회귀 결과
- `SDI_report_v2.md` — 최종 보고서